# Associative Recall Inference on Pretrained Models

This notebook runs **inference only** on existing pretrained checkpoints using the sequence-length benchmark pipeline with `task_variant="associative_recall"`.

In [ ]:
from __future__ import annotations

from pathlib import Path

import pandas as pd
from IPython.display import display

from pfns.utils import get_default_device
from pfns.experiments.model_benchmarks.analysis import (
    compute_mean_rank_tables,
    nested_metric_table_to_long_df,
)
from pfns.experiments.model_benchmarks.evaluation import evaluate_models_over_seqlens
from pfns.experiments.model_benchmarks.io import make_bundle_path, save_results_bundle
from pfns.experiments.model_benchmarks.model_registry import (
    get_autocast_models_from_registry,
    get_forward_models_from_registry,
    get_models_from_families,
    get_models_from_names,
)
from pfns.experiments.model_benchmarks.models import load_models_for_benchmark
import torch



In [ ]:
EXPERIMENT = {
    "name": "ar_inference_pretrained",
    "num_classes": 5,
    "num_features": 10,
    "num_test_samples": 100,
    "num_repetitions": 100,
    "data_generation_seed": 42,
    "use_warmup_iters": False,
    "print_timing": False,
    "seqlen_list": [128, 512, 2048, 8192],
    "task_variant": "associative_recall",
    "task_kwargs": {},
}

MODEL_SELECTION_MODE = "names"  # "names" | "families"

MODEL_NAMES = [
    "equal_params:Transformer_Comb_ST",
    "equal_params:DeltaNet_Comb_ST",
    "equal_params:GLA_Comb_ST",
]

MODEL_FAMILIES = ["equal_params"]

if MODEL_SELECTION_MODE == "names":
    models_to_compare = get_models_from_names(MODEL_NAMES)
else:
    models_to_compare = get_models_from_families(MODEL_FAMILIES)

print(f"Selected {len(models_to_compare)} model(s):")
for name in models_to_compare:
    print(" -", name)

display(pd.DataFrame.from_dict(models_to_compare, orient="index"))


In [ ]:
device = str(get_default_device())
print("Using device:", device)

models, configs = load_models_for_benchmark(models_to_compare, device=device)
autocast_models = get_autocast_models_from_registry(models_to_compare, device=device)
forward_models = get_forward_models_from_registry(models_to_compare)

print("Loaded models:", list(models.keys()))
print("Autocast models:", list(autocast_models.keys()))
print("Forward-only models:", forward_models)

In [ ]:
results = evaluate_models_over_seqlens(
    models=models,
    configs=configs,
    seqlen_list=EXPERIMENT["seqlen_list"],
    num_features=EXPERIMENT["num_features"],
    num_classes=EXPERIMENT["num_classes"],
    number_of_test_samples=EXPERIMENT["num_test_samples"],
    number_of_repetitions=EXPERIMENT["num_repetitions"],
    use_warmup_iters=EXPERIMENT["use_warmup_iters"],
    print_timing=EXPERIMENT["print_timing"],
    autocast_models=autocast_models,
    forward_models=forward_models,
    device=device,
    data_generation_seed=EXPERIMENT["data_generation_seed"],
    task_variant=EXPERIMENT["task_variant"],
    task_kwargs=EXPERIMENT["task_kwargs"],
    progress_desc="AR inference",
)

results["metadata"]

In [ ]:
metric_df = nested_metric_table_to_long_df(results["metric_table"])
timing_df = nested_metric_table_to_long_df(results["timing_table"])
memory_df = nested_metric_table_to_long_df(results["memory_table"])

print("metric_df:", metric_df.shape)
print("timing_df:", timing_df.shape)
print("memory_df:", memory_df.shape)

display(metric_df.head())
display(timing_df.head())
display(memory_df.head())

In [ ]:
metric_summary = (
    metric_df.groupby(["model", "metric"], observed=True)["value"]
    .agg(mean="mean", std="std")
    .reset_index()
)
display(metric_summary.sort_values(["metric", "mean"], ascending=[True, False]))

rank_tables = compute_mean_rank_tables(metric_df)
display(rank_tables["overall_ranks"].sort_values(["metric", "rank"]))

In [ ]:
import matplotlib.pyplot as plt

def _plot_metric_by_seqlen(df, metric_name, title, y_label):
    metric_slice = df[df["metric"] == metric_name].copy()
    if metric_slice.empty:
        print(f"No rows for metric={metric_name!r}.")
        return

    summary = (
        metric_slice.groupby(["model", "seqlen"], observed=True)["value"]
        .agg(mean="mean", std="std")
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    for model_name, g in summary.groupby("model", observed=True):
        g = g.sort_values("seqlen")
        x = g["seqlen"].to_numpy()
        y = g["mean"].to_numpy()
        y_std = g["std"].fillna(0.0).to_numpy()
        ax.plot(x, y, marker="o", label=model_name)
        ax.fill_between(x, y - y_std, y + y_std, alpha=0.2)

    ax.set_xscale("log")
    ax.set_xlabel("Sequence length")
    ax.set_ylabel(y_label)
    ax.set_title(title)

    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

_plot_metric_by_seqlen(metric_df, "acc", "Associative Recall Accuracy vs Sequence Length", "Accuracy")
_plot_metric_by_seqlen(metric_df, "ce", "Associative Recall CE vs Sequence Length", "Cross Entropy")
_plot_metric_by_seqlen(metric_df, "roc_auc", "Associative Recall ROC-AUC vs Sequence Length", "ROC-AUC")


In [ ]:
def _plot_system_by_seqlen(df, metric_name, title, y_label):
    metric_slice = df[df["metric"] == metric_name].copy()
    if metric_slice.empty:
        print(f"No rows for metric={metric_name!r}.")
        return

    summary = (
        metric_slice.groupby(["model", "seqlen"], observed=True)["value"]
        .agg(mean="mean", std="std")
        .reset_index()
    )

    fig, ax = plt.subplots(figsize=(9, 5))
    for model_name, g in summary.groupby("model", observed=True):
        g = g.sort_values("seqlen")
        x = g["seqlen"].to_numpy()
        y = g["mean"].to_numpy()
        y_std = g["std"].fillna(0.0).to_numpy()
        ax.plot(x, y, marker="o", label=model_name)
        ax.fill_between(x, y - y_std, y + y_std, alpha=0.2)

    ax.set_xscale("log")
    ax.set_xlabel("Sequence length")
    ax.set_ylabel(y_label)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

_plot_system_by_seqlen(timing_df, "forward_time_ms", "Forward Time vs Sequence Length", "Forward time (ms)")
_plot_system_by_seqlen(memory_df, "context_size_mb", "Context Size vs Sequence Length", "Context size (MB)")
_plot_system_by_seqlen(memory_df, "peak_allocated_mb", "Peak Allocated GPU Memory vs Sequence Length", "Peak allocated (MB)")
